# step 4 — 모델 다양성 (RQ3 인과: 지침 지시어 KV 치환, 504 통일, mean-pool)

**대응 RQ:** RQ3 **인과**, 모델 일반화. step4(Qwen)에서 '지침 레버 = Value, L27'을 봤다.
step2가 step1을 여러 모델로 확장했듯, 이건 step4를 **deepseek/granite/stable로 확장**해
'지침=Value'가 한 모델 특성인지 일반 원리인지 가른다.

**데이터셋:** step1/B/C와 **동일한 504 이름 풀 + 블록 커버**(42블록).

**정렬 단위 = `token_unit='mean'` (mean-pool, 핵심).** 지시어 `camelCase`/`snake_case`는 모델마다
토큰 수가 달라 `'all'`(전체 1:1)은 비Qwen에서 전부 스킵(nsub=0), `'last'`(마지막 1개)는 너무 약해
gap 있는 모델(stable)도 못 봤다. **mean-pool = 공여(반대지침) 토큰들을 평균 내 위반 지시어의**
**모든 토큰 자리에 넣는다.** 개수 불일치 허용(스킵 없음) + 단어 전체를 덮어(last보다 강함) →
모든 모델에서 whole-word급으로 측정 가능. (last 결과는 `results/step4_modeldiv_results/`에 보존.)

**gap 스크리닝.** 전이율 = (S_int−S_base)/(S_clean−S_base). 분모 gap=S_clean−S_base가 0에 가까우면
(지침을 텍스트로 바꿔도 행동이 안 변하면) 전이율은 정의 불가 → 그 모델·조건은 **'지침 레버 없음'**으로
분류하고 전이율 숫자는 버린다(granite ±거대값 노이즈 방지).

**결과가 뭐로 나오나:** 모델 × (방향·선행)마다 gap과 (층×kind) 전이율. 판정 —
- gap 충분 + **Value**로 전이 → 그 모델도 지침 레버=Value (Spotlight 반박 일반화)
- gap 충분 + **Key**로 전이 → 그 모델은 어텐션 경로 (예외)
- gap≈0 → 지침 레버 없음 (전이율 무의미, 별도 분류)

설계: `docs/step4/plan_model-diversity.md`. 원자료(불변,§6): `results/step4_modeldiv_mean/`.

> **메모리(T4):** output_attentions 안 씀(KV 편집만) → 가볍다. 모델은 한 개씩 로드→실행→해제.
> **재개:** 조건마다 저장, 이미 있으면 건너뜀(슬러그에 모델·블록·방향·선행·tok-mean 다 들어감).
> **규모:** 42블록×방향2×선행2×모델4 = 672 스윕. 무거우면 셀3의 BLOCKS를 줄이거나 모델을 하나씩.


In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)


In [ ]:
# 저장소 클론 및 브랜치 체크아웃
# [Llama 게이트 모델] meta-llama/Llama-3.2-3B는 라이선스 동의 + HF 로그인 필요.
#   huggingface.co에서 모델 라이선스 'Agree' 후 토큰 발급 -> 아래 주석 해제:
# from huggingface_hub import login; login('hf_...토큰...')
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step4/model-diversity
!git checkout step4/model-diversity
!git pull --quiet origin step4/model-diversity
!pip install -e . -q
import sys; sys.path.insert(0, 'src')


In [ ]:
# 조건 설정 — 4모델 x 42블록 x 방향2 x 선행2, token_unit='mean'(mean-pool, 스킵 없음).
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',        family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-1.3b-instruct', family='deepseek', dtype='float16'),
    ModelSpec(name='meta-llama/Llama-3.2-3B-Instruct',         family='llama',    dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',      family='stability',dtype='float16'),
]

DIRECTIONS = [Notation.CAMEL, Notation.SNAKE]   # camel->snake, snake->camel
PRECEDING_NC = [6, 0]                            # 균형 6/6, 전부위반 0/12
BLOCKS = list(range(42))                         # 504 이름 전부 커버(42블록). 무거우면 줄이기
TOKEN_UNIT = 'mean'                              # mean-pool: 공여 평균을 위반 전 자리에(스킵X, 강함)

def sweep_cond(model, target, nc, block):
    return Condition(
        model=model,
        preceding=PrecedingCode(n_compliant=nc, n_functions=12, composition=Composition.POOL,
                                pool_block=block),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target),
        intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers='sweep',
                                  target='instruction'),   # 지침 지시어 타깃(step4)
        seed=0, token_unit=TOKEN_UNIT,
    )

conds_by_model = {m.family: [sweep_cond(m, t, nc, b)
                             for t in DIRECTIONS for nc in PRECEDING_NC for b in BLOCKS]
                  for m in MODELS}
total = sum(len(v) for v in conds_by_model.values())
print('모델:', [m.family for m in MODELS])
print('모델당 조건:', len(next(iter(conds_by_model.values()))),
      '= 방향', len(DIRECTIONS), 'x 선행', len(PRECEDING_NC), 'x 블록', len(BLOCKS))
print('총 스윕:', total)

PREDICTION = ('예측(불확실): mean-pool은 whole-word급이라 Qwen은 Value 크게(~0.9급) 기대. stable은 '
              'gap 있으니 이제 Value/Key 구분 보일 것. deepseek/granite는 gap≈0이면 지침 레버 없음. '
              '예상과 달라도 조건 바꿔 맞추지 않고 그대로 기록(CLAUDE.md §4).')


In [ ]:
# 실행 — 모델 하나씩 로드->실행(재개)->해제. 조건마다 즉시 저장.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import gc, torch

STEP = 'step4_modeldiv_mean'   # last 결과(step4_modeldiv_results)와 분리 보존
for m in MODELS:
    conds = conds_by_model[m.family]
    todo = [c for c in conds if not result_path(c, step=STEP).exists()]
    print(f'[{m.family}] 조건 {len(conds)} / 남은 {len(todo)}')
    if not todo:
        print(f'  -> 이미 완료, 건너뜀'); continue
    handle = load_model(m)   # output_attentions 안 씀 -> eager 불필요
    print(f'  layers={handle.num_layers} GQA={handle.gqa_info()}')
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle)   # kind!=NONE + sweep + target=instruction -> 지침 스윕(mean)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ3', prediction=PREDICTION))
        if i % 20 == 0 or i == len(todo):
            e = out.metrics.extra
            print(f'    [{i}/{len(todo)}] nsub={e.get("n_substituted_tokens")} '
                  f'gap={e.get("S_clean",0)-e.get("S_base",0):+.2f}')
    del handle; gc.collect(); torch.cuda.empty_cache()
    print(f'  -> {m.family} 완료, 모델 해제')
print('전체 완료.')


In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
recs = []
for m in MODELS:
    for c in conds_by_model[m.family]:
        p = result_path(c, step='step4_modeldiv_mean')
        if p.exists(): recs.append(load_result(p))
print('로드:', len(recs), '-> results/step4_modeldiv_mean/')
# nsub 점검: mean은 위반 지시어의 토큰 수만큼(보통 2). 0이면 지시어 못 찾은 것.
import collections
ns = collections.Counter(r.metrics.extra.get('n_substituted_tokens') for r in recs)
print('n_substituted 분포:', dict(ns), '(위반 지시어 토큰 수. 0이 있으면 이상)')


In [ ]:
# 요약 — 모델 x (방향,선행): gap + (층xkind) 전이율 피크. gap 스크리닝.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict

KINDS = ['key','value','key_value']
GAP_MIN = 1.0   # |gap|<이 값이면 '지침 레버 없음'으로 분류(전이율 무의미)

agg = defaultdict(lambda: {k: defaultdict(list) for k in KINDS})
Sst = defaultdict(lambda: {'sb':[], 'sc':[]})
nlay = {}
for r in recs:
    c = r.condition; fam = c.model.family
    key = (fam, c.instruction.target_notation.value, c.preceding.n_compliant)
    Sst[key]['sb'].append(r.metrics.extra['S_base']); Sst[key]['sc'].append(r.metrics.extra['S_clean'])
    nlay[fam] = len(r.metrics.per_layer)
    for L, flat in r.metrics.per_layer.items():
        for k in KINDS:
            kk = f'{k}__recovery'
            if kk in flat: agg[key][k][int(L)].append(flat[kk])

def peak(key, k):
    c = agg[key][k]
    if not c: return None, None
    Ls = sorted(c); vs = [np.mean(c[L]) for L in Ls]
    i = int(np.argmax(np.abs(vs))); return Ls[i], float(vs[i])

def dlab(t): return 'camel->snake' if t=='camel' else 'snake->camel'
def plab(nc): return 'bal 6/6' if nc==6 else 'all-viol'

rows = []
for key in sorted(Sst):
    fam, t, nc = key
    sb = np.mean(Sst[key]['sb']); sc = np.mean(Sst[key]['sc']); gap = sc - sb
    Lv, v = peak(key, 'value'); Lk, kk = peak(key, 'key')
    lever = abs(gap) >= GAP_MIN
    rows.append({'model': fam, 'direction': dlab(t), 'preceding': plab(nc),
                 'gap': round(gap,2), 'lever?': 'Y' if lever else 'no(gap~0)',
                 'value_peak': round(v,2) if lever else None,
                 'key_peak': round(kk,2) if lever else None,
                 'peakL(V)': Lv if lever else None})
df = pd.DataFrame(rows)
print('=== 모델 x 조건: gap 스크리닝 + Value/Key 전이 피크 (token_unit=mean) ===')
print(df.to_string(index=False))

print('\n=== 모델별 요약 (gap>=%.1f 조건만) ===' % GAP_MIN)
for fam in [m.family for m in MODELS]:
    sub = [r for r in rows if r['model']==fam and r['lever?']=='Y']
    if not sub:
        print(f'  {fam}: 레버 있는 조건 없음(gap 전부 ~0) -> 지침 인과 불활성/측정불가'); continue
    vmean = np.mean([r['value_peak'] for r in sub]); kmean = np.mean([r['key_peak'] for r in sub])
    print(f'  {fam}: value {vmean:+.2f} / key {kmean:+.2f} (레버 조건 {len(sub)}/4) '
          f'-> {"Value 경로" if vmean>abs(kmean) else "불명확"}')

fams = [m.family for m in MODELS]
vmeans=[]; kmeans=[]; labs=[]
for fam in fams:
    sub=[r for r in rows if r['model']==fam and r['lever?']=='Y']
    labs.append(fam + ('' if sub else '\n(no lever)'))
    vmeans.append(np.mean([r['value_peak'] for r in sub]) if sub else 0)
    kmeans.append(np.mean([r['key_peak'] for r in sub]) if sub else 0)
x=np.arange(len(fams)); w=0.36
fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(x-w/2,kmeans,w,label='key (attention)',color='#2563C9')
ax.bar(x+w/2,vmeans,w,label='value (content)',color='#C6552B')
ax.axhline(0,color='#666',lw=.8); ax.set_xticks(x); ax.set_xticklabels(labs,fontsize=9)
ax.set_ylabel('mean peak transition (mean-pool, lever conds)')
ax.set_title('step4 model diversity — is the instruction lever Value across models? (gap-screened, mean-pool)',fontsize=10)
ax.legend(fontsize=9.5); ax.grid(axis='y',alpha=.15)
fig.tight_layout(); plt.savefig('step4_modeldiv_mean_summary.png',dpi=140,bbox_inches='tight'); plt.show()
print('저장: step4_modeldiv_mean_summary.png')


In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive('step4_modeldiv_mean_results', 'zip', 'results/step4_modeldiv_mean')
try:
    from google.colab import files
    files.download('step4_modeldiv_mean_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): step4_modeldiv_mean_results.zip', e)
